# Pretraining

This notebook uses the bundled scikit-learn Digits dataset as a self-supervised reconstruction task. There is no supervised label. Instead, the model learns by masking pixel intensities and reconstructing them from row, column, and surrounding context.

Only local dependencies are used. The dataset comes from scikit-learn, so the notebook remains self-contained and does not download data.

In [1]:
import contextlib
import io
import logging

import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from sklearn.datasets import load_digits

import json2vec as j2v

logger.remove()

Each image is converted into a nested `pixels` array. Every pixel is represented as a small JSON object with `row`, `column`, and `intensity`, which is closer to the nested records JSON2Vec is designed to handle.

In [2]:
digits = load_digits()

records = pl.DataFrame(
    [
        {
            "pixels": [
                {"row": row, "column": column, "intensity": float(image[row, column]) / 16.0}
                for row in range(8)
                for column in range(8)
            ]
        }
        for image in digits.images[:24]
    ],
    strict=False,
)

records.head()

pixels
list[struct[3]]
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]"
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]"
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]"
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]"
"[{0,0,0.0}, {0,1,0.0}, … {7,7,0.0}]"


The `pixels` array creates a local context encoder. Row and column are categorical position hints with light masking, and intensity is numeric content with `p_mask=0.50`, so random intensities are hidden and reconstructed during training.

In [3]:
model = j2v.Model.from_schema(
    j2v.Array(
        j2v.Category("row", query="[*].pixels[*].row", max_vocab_size=8),
        j2v.Category("column", query="[*].pixels[*].column", max_vocab_size=8),
        j2v.Number("intensity", query="[*].pixels[*].intensity"),
        name="pixels",
        max_length=64,
    ),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

_ = model.set(j2v.where("name") == "intensity", p_mask=0.50)
_ = model.set(j2v.where("type") == "category", p_mask=0.05)

The data module streams the nested records through the model schema. No target field is required because masking supplies the training signal.

In [4]:
datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=16,
    chunk_batch_size=32,
    sample_rate=1.0,
)

A single epoch is enough for the documentation example. The important behavior is that the same training loop works whether the task is supervised or mask-based pretraining.

In [5]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    logging.disable(logging.CRITICAL)
    try:
        lit.seed_everything(7, workers=True, verbose=False)
        trainer = lit.Trainer(
            accelerator="cpu",
            max_epochs=1,
            logger=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            enable_checkpointing=False,
            limit_train_batches=1,
            limit_val_batches=1,
        )
        trainer.fit(model=model, datamodule=datamodule)
    finally:
        logging.disable(logging.NOTSET)

The plot shows a root record encoder, a nested pixel encoder, and the masked numeric intensity field that drives the pretraining loss.

In [6]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  27,765                                │
│  Schema map                                                            arrays  2                                     │
│                                                                        fields  3                                     │
│                                                                       targets  0                                     │
│                                                                        embeds  0                                     │
│                               